# In Class 4/28
## Bayes Models
### Dirks Wright

In [1]:
### load packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sweetviz as sv
### packages for gausian naive bayes
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier



In [2]:
### load data
df = pd.read_csv('Kaggle_Comp/train.csv')
df.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [3]:
### sweetviz report
report = sv.analyze(df)
report.show_html('4_28_report.html')

                                             |          | [  0%]   00:00 -> (? left)

Report 4_28_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [4]:
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score, precision_score, recall_score, f1_score

In [15]:
df.select_dtypes(include=["object", "category"]).columns

Index(['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
       'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region',
       'Irrigation_Need', 'target_name'],
      dtype='object')

In [16]:
### preprocessing dummify categorical features
df = pd.get_dummies(df, columns=['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region'], drop_first=True)


In [17]:
### train test split
df['target'] = (df['Irrigation_Need'] == "High").astype(int)
df['target_name'] = df['target'].map({0:'Low/Medium', 1:'High'})

X = df.drop(['id', 'Irrigation_Need', 'target', 'target_name'], axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y, random_state=222)


### Functions

In [7]:
from sklearn.model_selection import cross_validate

In [9]:
# helper function for cross validation comparison
def compare_models_cv(models,X_train,y_train,cv):
    cv_results=[]
    for name,model in models.items():
        scores=cross_validate(model,X_train,y_train,cv=cv,scoring=["accuracy","precision","recall","f1","balanced_accuracy"])
        cv_results.append({
            "model":name,
            "cv_accuracy":scores["test_accuracy"].mean(),
            "cv_precision":scores["test_precision"].mean(),
            "cv_recall":scores["test_recall"].mean(),
            "cv_f1":scores["test_f1"].mean(),
            "cv_balanced_accuracy":scores["test_balanced_accuracy"].mean()
        })
    return pd.DataFrame(cv_results).sort_values("cv_balanced_accuracy",ascending=False)

In [10]:
# helper function for threshold tuning
def evaluate_thresholds(y_true,probs,thresholds=None):
    if thresholds is None:
        thresholds=np.linspace(0.1,0.9,17)
    rows=[]
    for t in thresholds:
        preds=(probs[:,1]>=t).astype(int)
        rows.append({
            "threshold":t,
            "precision":precision_score(y_true,preds,zero_division=0),
            "recall":recall_score(y_true,preds,zero_division=0),
            "f1":f1_score(y_true,preds,zero_division=0),
            "balanced_accuracy":balanced_accuracy_score(y_true,preds)
        })
    return pd.DataFrame(rows)

In [11]:
# helper function for held out test evaluation
def evaluate_models_test(models,X_train,X_test,y_train,y_test):
    trained_models={}
    test_results=[]
    for name,model in models.items():
        model.fit(X_train,y_train)
        trained_models[name]=model
        preds=model.predict(X_test)
        test_results.append({
            "model":name,
            "test_accuracy":accuracy_score(y_test,preds),
            "test_precision":precision_score(y_test,preds,zero_division=0),
            "test_recall":recall_score(y_test,preds,zero_division=0),
            "test_f1":f1_score(y_test,preds,zero_division=0),
            "test_balanced_accuracy":balanced_accuracy_score(y_test,preds)
        })
    return trained_models,pd.DataFrame(test_results).sort_values("test_balanced_accuracy",ascending=False)

### Models & CV

In [12]:
from sklearn.model_selection import StratifiedKFold

In [18]:
### create pipeline
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=222)
models = {"Naive Bayes": GaussianNB(),
            "Logistic Regression": LogisticRegression(max_iter=1000, random_state=222),
            "Random Forest": RandomForestClassifier(random_state=222)}

### Model Comparison w/ cross val

In [19]:
compare_models_cv(models,X_train,y_train,cv)

c:\Users\Dirks Wright\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Dirks Wright\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regres

,model,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_balanced_accuracy
2,Random Forest,0.995696,0.974848,0.894032,0.932681,0.946618
0,Naive Bayes,0.956262,0.423142,0.856905,0.566513,0.908297
1,Logistic Regression,0.979657,0.774947,0.549592,0.643015,0.772043


In [20]:
### fit models on full train data
trained_models, test_results = evaluate_models_test(
models,X_train,X_test,y_train,y_test
)
test_results

c:\Users\Dirks Wright\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,model,test_accuracy,test_precision,test_recall,test_f1,test_balanced_accuracy
2,Random Forest,0.996032,0.974129,0.905045,0.938317,0.952108
0,Naive Bayes,0.956929,0.428068,0.867444,0.573248,0.913730
1,Logistic Regression,0.979968,0.774004,0.564017,0.652533,0.779168


### Threshold

In [21]:
### threshold logistic regression
model = trained_models['Logistic Regression']
probs_lr = model.predict_proba(X_test)

threshold_results_lr = evaluate_thresholds(y_test, probs_lr)
best = threshold_results_lr.sort_values("balanced_accuracy", ascending=False).iloc[0]

default_preds = (probs_lr[:,1]>=0.5).astype(int)

print(threshold_results_lr.sort_values("balanced_accuracy", ascending=False))
print()
print('Default Threshold:', 0.5)
print('Default Balanced Accuracy:', round(balanced_accuracy_score(y_test, default_preds), 4))
print('Best Threshold:', round(best['threshold'], 4))
print('Best Balanced Accuracy:', round(best['balanced_accuracy'], 4))

    threshold  precision    recall        f1  balanced_accuracy
0        0.10   0.407768  0.866968  0.554659           0.911764
1        0.15   0.476896  0.827701  0.605133           0.898189
2        0.20   0.536004  0.786530  0.637539           0.881520
3        0.25   0.587698  0.750357  0.659141           0.866098
4        0.30   0.633791  0.714184  0.671590           0.849974
5        0.35   0.673527  0.677535  0.675525           0.833102
6        0.40   0.708079  0.636126  0.670177           0.813539
7        0.45   0.743946  0.599476  0.663943           0.796179
8        0.50   0.774004  0.564017  0.652533           0.779168
9        0.55   0.805294  0.528558  0.638218           0.762074
10       0.60   0.839446  0.490243  0.618990           0.743504
11       0.65   0.864338  0.453356  0.594755           0.725450
12       0.70   0.894326  0.408853  0.561163           0.703593
13       0.75   0.915009  0.361257  0.518000           0.680049
14       0.80   0.935391  0.310090  0.46

In [22]:
### threshold naive bayes
model_nb = trained_models['Naive Bayes']
probs_nb = model_nb.predict_proba(X_test)
threshold_results_nb = evaluate_thresholds(y_test, probs_nb)
best_nb = threshold_results_nb.sort_values("balanced_accuracy", ascending=False).iloc[0]
default_preds_nb = (probs_nb[:,1]>=0.5).astype(int)
print(threshold_results_nb.sort_values("balanced_accuracy", ascending=False))
print()
print('Default Threshold:', 0.5)
print('Default Balanced Accuracy:', round(balanced_accuracy_score(y_test, default_preds_nb), 4))
print('Best Threshold:', round(best_nb['threshold'], 4))
print('Best Balanced Accuracy:', round(best_nb['balanced_accuracy'], 4))



    threshold  precision    recall        f1  balanced_accuracy
4        0.30   0.334282  0.894574  0.486696           0.916556
3        0.25   0.312284  0.901475  0.463875           0.916493
5        0.35   0.356180  0.886721  0.508218           0.915712
6        0.40   0.378644  0.881009  0.529652           0.915566
2        0.20   0.288036  0.906949  0.437217           0.914804
7        0.45   0.401113  0.874584  0.549985           0.914767
8        0.50   0.428068  0.867444  0.573248           0.913730
1        0.15   0.263757  0.914802  0.409459           0.913353
9        0.55   0.456362  0.858639  0.595970           0.911675
0        0.10   0.235451  0.922418  0.375145           0.909541
10       0.60   0.483625  0.846978  0.615691           0.907889
11       0.65   0.513693  0.830319  0.634710           0.901600
12       0.70   0.550339  0.811756  0.655962           0.894437
13       0.75   0.591289  0.791528  0.676911           0.886326
14       0.80   0.640167  0.768444  0.69

In [23]:
### threshold random forest
model_rf = trained_models['Random Forest']
probs_rf = model_rf.predict_proba(X_test)
threshold_results_rf = evaluate_thresholds(y_test, probs_rf)
best_rf = threshold_results_rf.sort_values("balanced_accuracy", ascending=False).iloc[0]
default_preds_rf = (probs_rf[:,1]>=0.5).astype(int)
print(threshold_results_rf.sort_values("balanced_accuracy", ascending=False))
print()
print('Default Threshold:', 0.5)
print('Default Balanced Accuracy:', round(balanced_accuracy_score(y_test, default_preds_rf), 4))
print('Best Threshold:', round(best_rf['threshold'], 4))
print('Best Balanced Accuracy:', round(best_rf['balanced_accuracy'], 4))


    threshold  precision    recall        f1  balanced_accuracy
0        0.10   0.661267  0.961685  0.783671           0.972345
1        0.15   0.832286  0.945978  0.885498           0.969701
2        0.20   0.876248  0.940267  0.907129           0.967843
3        0.25   0.912953  0.935983  0.924324           0.966452
4        0.30   0.933635  0.934079  0.933857           0.965894
5        0.35   0.946757  0.930985  0.938805           0.964589
6        0.40   0.958796  0.924798  0.941490           0.961713
7        0.45   0.967839  0.916706  0.941579           0.957828
8        0.50   0.972690  0.906949  0.938670           0.953035
9        0.55   0.978458  0.897192  0.936065           0.948255
10       0.60   0.981437  0.880771  0.928383           0.940098
11       0.65   0.983941  0.860305  0.917979           0.929910
12       0.70   0.985552  0.827939  0.899897           0.913760
13       0.75   0.986168  0.797477  0.881842           0.898546
14       0.80   0.987995  0.724655  0.83

### Optimal Thresholds

In [25]:
preds_lr = (trained_models['Logistic Regression'].predict_proba(X_test)[:,1]>= 0.1).astype(int)
print('Logistic Regression')
print('precision:' , round(precision_score(y_test,preds_lr,zero_division=0),4))
print('recall:' , round(recall_score(y_test,preds_lr,zero_division=0),4) )
print('f1:' , round(f1_score(y_test,preds_lr,zero_division=0),4) )
print('balanced_accuracy_score:' , round(balanced_accuracy_score(y_test,preds_lr),4) )
print()

preds_nb = (trained_models['Naive Bayes'].predict_proba(X_test)[:,1]>= 0.3).astype(int)
print('Naive Bayes')
print('precision:' , round(precision_score(y_test,preds_nb,zero_division=0),4))
print('recall:' , round(recall_score(y_test,preds_nb,zero_division=0),4) )
print('f1:' , round(f1_score(y_test,preds_nb,zero_division=   0),4) )
print('balanced_accuracy_score:' , round(balanced_accuracy_score(y_test,preds_nb),4) )
print() 

preds_rf = (trained_models['Random Forest'].predict_proba(X_test)[:,1]>= 0.1).astype(int)
print('Random Forest')
print('precision:' , round(precision_score(y_test,preds_rf,zero_division=0),4))
print('recall:' , round(recall_score(y_test,preds_rf,zero_division=0),4) )
print('f1:' , round(f1_score(y_test,preds_rf,zero_division=   0),4) )
print('balanced_accuracy_score:' , round(balanced_accuracy_score(y_test,preds_rf),4) )
print() 

Logistic Regression
precision: 0.4078
recall: 0.867
f1: 0.5547
balanced_accuracy_score: 0.9118

Naive Bayes
precision: 0.3343
recall: 0.8946
f1: 0.4867
balanced_accuracy_score: 0.9166

Random Forest
precision: 0.6613
recall: 0.9617
f1: 0.7837
balanced_accuracy_score: 0.9723



In [26]:
### classification report
print('Logistic Regression')
print(classification_report(y_test, preds_lr, zero_division=0))
print('Naive Bayes')
print(classification_report(y_test, preds_nb, zero_division=0))
print('Random Forest')
print(classification_report(y_test, preds_rf, zero_division=0))


Logistic Regression
              precision    recall  f1-score   support

           0       1.00      0.96      0.98    121798
           1       0.41      0.87      0.55      4202

    accuracy                           0.95    126000
   macro avg       0.70      0.91      0.77    126000
weighted avg       0.98      0.95      0.96    126000

Naive Bayes
              precision    recall  f1-score   support

           0       1.00      0.94      0.97    121798
           1       0.33      0.89      0.49      4202

    accuracy                           0.94    126000
   macro avg       0.67      0.92      0.73    126000
weighted avg       0.97      0.94      0.95    126000

Random Forest
              precision    recall  f1-score   support

           0       1.00      0.98      0.99    121798
           1       0.66      0.96      0.78      4202

    accuracy                           0.98    126000
   macro avg       0.83      0.97      0.89    126000
weighted avg       0.99     

### Interpretation

I selected high irrigation need as Class 1. I focused on this class because it is important to correctly identify fields that need a high amount of irrigation, and this is also the least common of the 3 classes. For Naive Bayes, I changed the threshold from the default 0.50 to 0.30. This slightly improved balanced accuracy from baseline.The Naive Bayes model had a default balanced accuracy of 0.9137 at the 0.50 threshold. After lowering the threshold to 0.30, the balanced accuracy increased slightly to 0.9166. Recall for the High irrigation class improved from 0.8674 to 0.8946, meaning the model found more of the true High irrigation cases. However, precision decreased from 0.4281 to 0.3343, and the F1-score decreased from 0.5732 to 0.4867.

The main tradeoff was between recall and precision. Lowering the threshold made the model more likely to predict High irrigation need, which helped it catch more true High cases, but it also created more false positives. The model became less strict, so it improved recall but lowered precision.

Compared with the other models after threshold tuning, Naive Bayes had slightly better recall than Logistic Regression (0.89 compared with 0.87) and a slightly higher balanced accuracy (0.9166 compared with 0.9118). However, Naive Bayes had lower precision and F1-score than Logistic Regression. Random Forest performed best overall, with precision of 0.66, recall of 0.96, F1-score of 0.78, and balanced accuracy of 0.9723 for the High irrigation class. Overall, Naive Bayes is useful when recall is the main priority, but Random Forest gives the strongest balance across the classification report.
